## Logistic Regression

In [ ]:
import sys

sys.path.append("..")

In [ ]:
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

In [ ]:
from src.data import load_data, preprocess_data
from src.features import prepare_classical_features
from src.splits import data_splitting

In [ ]:
df = load_data()
df = preprocess_data(df)
X, y = prepare_classical_features(df)
X_train, X_test, y_train, y_test = data_splitting(X, y)

### Model Training
Before we train any model, since we have low data, around 80 training samples, we will use Startified KFold Cross evaluation, to evaluate the models along k different folds. 

In [ ]:
print(X_train, X_train.shape)
print(X_test, X_test.shape)
print(y_train, y_train.shape)
print(y_test, y_test.shape)

In [ ]:
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=13)
skf.get_n_splits()
splits = list(skf.split(X_train, y_train))

In [ ]:
splits

In [ ]:
lgr = LogisticRegression(random_state=13)

Since we are doing kfold cross validation, let us have an array for aoc and matrix to track confusion matrices

In [ ]:
lgr_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": [],
}
lgr_cm = np.zeros((2, 2), dtype=int)

In [ ]:
print(type(X_train))
print(type(y_train))
print(type(X_test))
print(type(y_test))

We need to convert y into numpy arrays for consistency and easier manipulation

In [ ]:
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [ ]:
for i, (train_idx, test_idx) in enumerate(splits):
    X_fold_train, y_fold_train = X_train[train_idx], y_train[train_idx]
    X_fold_test, y_fold_test = X_train[test_idx], y_train[test_idx]
    
    lgr_cls = lgr.fit(X_fold_train, y_fold_train)
    y_pred = lgr_cls.predict(X_fold_test)
    y_scores = lgr_cls.predict_proba(X_fold_test)[:, 1]
    
    auc = metrics.roc_auc_score(y_true=y_fold_test, y_score=y_scores)
    lgr_metrics["auc"].append(auc)
    
    lgr_metrics["accuracy"].append(metrics.accuracy_score(y_true=y_fold_test, y_pred=y_pred))
    lgr_metrics["precision"].append(metrics.precision_score(y_true=y_fold_test, y_pred=y_pred))
    lgr_metrics["recall"].append(metrics.recall_score(y_true=y_fold_test, y_pred=y_pred))
    lgr_metrics["f1"].append(metrics.f1_score(y_true=y_fold_test, y_pred=y_pred))

    cm = metrics.confusion_matrix(y_true=y_fold_test, y_pred=y_pred)
    lgr_cm += cm

In [ ]:
print(lgr_metrics)

In [ ]:
table = pd.DataFrame(index=lgr_metrics.keys(), columns=["cv_metric"])
for key, metric in lgr_metrics.items():
    table.loc[key, "cv_metric"] = f"{np.mean(metric):.3f} ± {np.std(metric):.3f}"

Validating our calculation

In [ ]:
table